In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.colors as plc
import astropy.units as u
import astropy.constants as c
from astropy.io import fits
from astropy.time import Time
from astropy.modeling.functional_models import Voigt1D
from scipy.optimize import curve_fit, minimize
from sunburn import analysis, tools, hst_observation, spectroscopy, systematics
import os
import emcee

%config InlineBackend.figure_format = 'retina'

pylab.rcParams['figure.figsize'] = 9.0,6.5
pylab.rcParams['font.size'] = 18

In [ ]:
datasets_visit1 = ['oe8ta1010', 
                   'oe8ta1020', 'oe8ta1030', 'oe8ta1040', 'oe8ta1050']
datasets_visit2 = ['oen502030', 'oen502040', 'oen502050']
datasets_visit3 = ['oen501010', 'oen501020', 'oen501030', 'oen501040', 'oen501050']
datasets_visit4 = ['oen552010', 'oen552020', 'oen552030', 'oen552040', 'oen552050']

visit1 = hst_observation.Visit(datasets_visit1, 'stis', prefix='../data/v1/')
visit2 = hst_observation.Visit(datasets_visit2, 'stis', prefix='../data/v2/')
visit3 = hst_observation.Visit(datasets_visit3, 'stis', prefix='../data/v3/')
visit4 = hst_observation.Visit(datasets_visit4, 'stis', prefix='../data/v4/')

In [ ]:
visit1.assign_splits('../data/v1/splits10/')
visit2.assign_splits('../data/v2/splits10/')
visit3.assign_splits('../data/v3/splits10/')
visit4.assign_splits('../data/v4/splits10/')

In [ ]:
# Fit Doppler shift using a Voigt profile
def v_func(v, center, amplitude, fwhm_L, fwhm_G):
    v_shifted = v - center
    v_f = Voigt1D(amplitude_L=amplitude, fwhm_L=fwhm_L, fwhm_G=fwhm_G)
    return v_f(v_shifted)

# Log-likelihood function
def log_likelihood(theta, x, y, yerr):
    v0, a, fl, fg = theta
    model = v_func(x, v0, a, fl, fg)
    sigma2 = yerr ** 2
    return -0.5 * np.sum((y - model) ** 2 / sigma2 + np.log(sigma2))

nll = lambda *args: -log_likelihood(*args)
cs = c.c.to(u.km / u.s).value

# Priors
def log_prior(theta):
    v0, a, fl, fg = theta
    if -30.0 < v0 < 30 and 0.0 < a < 1.0 and 20 < fl < 100 and 0 < fg < 100:
        return 0.0
    return -np.inf

# Log-probability function
def log_probability(theta, x, y, yerr):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, x, y, yerr)

# Read spectrum from orbit object
def get_spec(orbit, w0, v_range):
    v_all = (orbit.wavelength - w0) / w0 * cs
    i0 = tools.nearest_index(v_all, v_range[0])
    i1 = tools.nearest_index(v_all, v_range[1])
    v = v_all[i0:i1 + 1]
    f = orbit.flux[i0:i1 + 1]
    f_unc = orbit.error[i0:i1 + 1]
    return v, f, f_unc

In [ ]:
# Fit the wavelength shift in a strong emission line
w0 = 1238.821
v_range = np.array([-1000, 1000])
scale = 1E-13

# Fit function
def fit_v0(exposure, p0, nsteps=1000, plot_corner=True):
    v, f, f_unc = get_spec(exposure, w0, v_range)
    f_unc[np.where(f_unc < 1E-15)] = np.median(f_unc)
    soln = minimize(nll, p0, args=(v, f / scale, f_unc / scale), method='Nelder-Mead')
    pos = soln.x + 1e-3 * np.random.randn(16, 4)
    nwalkers, ndim = pos.shape

    sampler = emcee.EnsembleSampler(
        nwalkers, ndim, log_probability, args=(v, f / scale, f_unc / scale), #pool=pool
        )
    sampler.run_mcmc(pos, nsteps, progress=True)

    labels = ['v0', 'ampl', 'fl', 'fg']
    flat_samples = sampler.get_chain(discard=300, thin=15, flat=True)
    if plot_corner is True:
        fig = corner.corner(flat_samples, labels=labels)
    mcmc = np.percentile(flat_samples[:, 0], [16, 50, 84])
    q = np.diff(mcmc)
    print('Measured velocity shift = $%.2f^{+%.2f}_{-%.2f}$ km/s' %(mcmc[1], q[1], q[0]))
    return mcmc[1], q[0], q[1]

In [ ]:
w0 = 1238.821
v_range = np.array([-1000, 1000])
scale = 1E-13

p0 = (-15, 0.23, 80, 5)

results_v1 = []

for o in visit1.orbit:
    exp = visit1.orbit[o]
    v0_fit, v0_ll, v0_ul = fit_v0(exp, p0, nsteps=2000, plot_corner=False)
    results_v1.append([v0_fit, v0_ll, v0_ul])

In [ ]:
p0 = (0, 0.23, 80, 5)

results_v2 = []

for o in visit2.orbit:
    exp = visit2.orbit[o]
    v0_fit, v0_ll, v0_ul = fit_v0(exp, p0, nsteps=2000, plot_corner=False)
    results_v2.append([v0_fit, v0_ll, v0_ul])

In [ ]:
p0 = (0, 0.23, 80, 5)

results_v3 = []

for o in visit3.orbit:
    exp = visit3.orbit[o]
    v0_fit, v0_ll, v0_ul = fit_v0(exp, p0, nsteps=2000, plot_corner=False)
    results_v3.append([v0_fit, v0_ll, v0_ul])

In [ ]:
p0 = (0, 0.23, 80, 5)

results_v4 = []

for o in visit4.orbit:
    exp = visit4.orbit[o]
    v0_fit, v0_ll, v0_ul = fit_v0(exp, p0, nsteps=2000, plot_corner=False)
    results_v4.append([v0_fit, v0_ll, v0_ul])

In [ ]:
times_v1 = []
t_span_v1 = []
for o in visit1.orbit:
    times_v1.append((visit1.orbit[o].end_JD.jd + visit1.orbit[o].start_JD.jd) / 2)
    t_span_v1.append((visit1.orbit[o].end_JD.jd - visit1.orbit[o].start_JD.jd) / 2)
times_v1 = np.array(times_v1)
t_span_v1 = np.array(t_span_v1)

times_v2 = []
t_span_v2 = []
for o in visit2.orbit:
    times_v2.append((visit2.orbit[o].end_JD.jd + visit2.orbit[o].start_JD.jd) / 2)
    t_span_v2.append((visit2.orbit[o].end_JD.jd - visit2.orbit[o].start_JD.jd) / 2)
times_v2 = np.array(times_v2)
t_span_v2 = np.array(t_span_v2)
    
times_v3 = []
t_span_v3 = []
for o in visit3.orbit:
    times_v3.append((visit3.orbit[o].end_JD.jd + visit3.orbit[o].start_JD.jd) / 2)
    t_span_v3.append((visit3.orbit[o].end_JD.jd - visit3.orbit[o].start_JD.jd) / 2)
times_v3 = np.array(times_v3)
t_span_v3 = np.array(t_span_v3)
    
times_v4 = []
t_span_v4 = []
for o in visit4.orbit:
    times_v4.append((visit4.orbit[o].end_JD.jd + visit4.orbit[o].start_JD.jd) / 2)
    t_span_v4.append((visit4.orbit[o].end_JD.jd - visit4.orbit[o].start_JD.jd) / 2)
times_v4 = np.array(times_v4)
t_span_v4 = np.array(t_span_v4)

In [ ]:
results_v1 = np.array(results_v1)
results_v2 = np.array(results_v2)
results_v3 = np.array(results_v3)
results_v4 = np.array(results_v4)

fig, ax = plt.subplots(nrows=2, ncols=2, sharey=True)

ax[0, 0].errorbar(times_v1 - 2.459E6, results_v1[:, 0], 
                  yerr=(results_v1[:, 1], results_v1[:, 2]), 
                  #xerr=t_span_v1,
                  fmt='o', color='k', markersize=10, markeredgecolor='white')
ax[0, 1].errorbar(times_v2 - 2.459E6, results_v2[:, 0], 
                  yerr=(results_v2[:, 1], results_v2[:, 2]), 
                  fmt='o', color='k', markersize=10, markeredgecolor='white')
ax[1, 0].errorbar(times_v3 - 2.459E6, results_v3[:, 0], 
                  yerr=(results_v3[:, 1], results_v3[:, 2]), 
                  fmt='o', color='k', markersize=10, markeredgecolor='white')
ax[1, 1].errorbar(times_v4 - 2.459E6, results_v4[:, 0], 
                  yerr=(results_v4[:, 1], results_v4[:, 2]), 
                  fmt='o', color='k', markersize=10, markeredgecolor='white')
ax[0, 0].set_title('Visit A', fontsize=15)
ax[0, 1].set_title('Visit B', fontsize=15)
ax[1, 0].set_title('Visit C', fontsize=15)
ax[1, 1].set_title('Visit D', fontsize=15)
fig.supylabel(r'Doppler velocity shift (km s$^{-1}$)')
fig.supxlabel(r'Julian Date $-$ 2.459E6')
plt.tight_layout()
# plt.savefig('rv_shifts.pdf')


In [ ]:
# Save results in a text file
array = np.concatenate([results_v1, results_v2, results_v3, results_v4])
np.savetxt('rv_shifts.dat', array)